# Variant Risk Explainer: DNABERT-2 Fine-tuning

This notebook trains from the sequence CSV files produced by `01_prepare_clinvar_dataset.ipynb`.

Research and education only. This does not produce a clinically validated diagnostic model.

## 1. Runtime

In Colab, choose `Runtime -> Change runtime type -> GPU` before running training.

In [ ]:
!python --version
!nvidia-smi || true

## 2. Find Repository Root

Clone or upload the full repository first. The notebook will search `/content` for the training script, so the GitHub repo folder name can be different.

In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    candidates = [Path.cwd(), Path('/content/variant-risk-explainer')]
    for candidate in candidates:
        script = candidate / 'training' / 'scripts' / 'train_dnabert2_classifier.py'
        if script.exists():
            return candidate

    content_root = Path('/content')
    if content_root.exists():
        matches = list(content_root.glob('*/training/scripts/train_dnabert2_classifier.py'))
        if matches:
            return matches[0].parents[2]

    raise FileNotFoundError(
        'Could not find training/scripts/train_dnabert2_classifier.py. Clone or upload the full repository first.'
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%cd {PROJECT_ROOT}
print(f'Project root: {PROJECT_ROOT}')

## 3. Install Dependencies

This installs the Colab training dependencies. If Colab asks you to restart the runtime after installation, restart and run the notebook again from the top.

In [ ]:
!pip install -q -r training/requirements-colab.txt

## 4. Locate Sequence CSV Files

Upload your CSVs into one of these locations before running this cell:

- `training/csv_files/`
- `data/processed/`
- `training/data/processed/`

Required file names:

- `train_with_sequences.csv`
- `val_with_sequences.csv`
- `test_with_sequences.csv`

In [ ]:
import pandas as pd

CSV_SEARCH_DIRS = [
    PROJECT_ROOT / 'training' / 'csv_files',
    PROJECT_ROOT / 'data' / 'processed',
    PROJECT_ROOT / 'training' / 'data' / 'processed',
]


def find_csv(filename: str) -> Path:
    for directory in CSV_SEARCH_DIRS:
        candidate = directory / filename
        if candidate.exists():
            return candidate
    searched = '\n'.join(str(directory / filename) for directory in CSV_SEARCH_DIRS)
    raise FileNotFoundError(f'Could not find {filename}. Searched:\n{searched}')


TRAIN_CSV = find_csv('train_with_sequences.csv')
VAL_CSV = find_csv('val_with_sequences.csv')
TEST_CSV = find_csv('test_with_sequences.csv')

print(f'Train CSV: {TRAIN_CSV}')
print(f'Val CSV:   {VAL_CSV}')
print(f'Test CSV:  {TEST_CSV}')

## 5. Check CSV Quality

This checks required columns, sequence lengths, bad sequence characters, and labels. The training script will also defensively drop uncertain, conflicting, risk, association, drug-response, protective, and not-provided CLNSIG rows.

In [ ]:
from training.utils.label_utils import assign_binary_label

REQUIRED_COLUMNS = {'sequence', 'label'}


def inspect_split(name: str, path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f'{name} is missing columns: {sorted(missing)}')

    bad_sequence_chars = int((~df['sequence'].astype(str).str.upper().str.match(r'^[ACGTN]+$')).sum())
    cleanable_rows = None
    if 'CLNSIG' in df.columns:
        clean_labels = df['CLNSIG'].apply(assign_binary_label)
        cleanable_rows = int(clean_labels.notna().sum())

    print(f'\n{name}')
    print(f'rows: {len(df):,}')
    print(f'label counts: {df["label"].value_counts(dropna=False).sort_index().to_dict()}')
    print(
        'sequence length min/mean/max:',
        int(df['sequence'].astype(str).str.len().min()),
        round(float(df['sequence'].astype(str).str.len().mean()), 2),
        int(df['sequence'].astype(str).str.len().max()),
    )
    print(f'bad sequence rows: {bad_sequence_chars:,}')
    if cleanable_rows is not None:
        print(f'rows after defensive CLNSIG cleaning: {cleanable_rows:,}')
    display(df.head(3))
    return df


train_preview = inspect_split('train', TRAIN_CSV)
val_preview = inspect_split('val', VAL_CSV)
test_preview = inspect_split('test', TEST_CSV)

## 6. Training Settings

Use conservative settings first. Increase epochs or batch size later if the run is stable. For small sample CSVs, expect overfitting; this is only a research demo.

If you only want to test the notebook on CPU, set `CPU_SMOKE_TEST = True`. That uses a tiny model and a small number of rows so you can see progress without waiting for DNABERT-2 CPU training.

In [ ]:
CPU_SMOKE_TEST = False

if CPU_SMOKE_TEST:
    MODEL_NAME = 'hf-internal-testing/tiny-random-bert'
    OUTPUT_DIR = PROJECT_ROOT / 'training' / 'output' / 'cpu-smoke-test'
    MAX_LENGTH = 128
    EPOCHS = 1
    BATCH_SIZE = 8
    LEARNING_RATE = 5e-5
    MAX_TRAIN_SAMPLES = 64
    MAX_VAL_SAMPLES = 32
    MAX_TEST_SAMPLES = 32
    FORCE_CPU = True
else:
    MODEL_NAME = 'zhihan1996/DNABERT-2-117M'
    OUTPUT_DIR = PROJECT_ROOT / 'training' / 'output' / 'dnabert2-clinvar-grch38'
    MAX_LENGTH = 512
    EPOCHS = 1
    BATCH_SIZE = 4
    LEARNING_RATE = 2e-5
    MAX_TRAIN_SAMPLES = 0
    MAX_VAL_SAMPLES = 0
    MAX_TEST_SAMPLES = 0
    FORCE_CPU = False

MIN_SEQUENCE_LENGTH = 200
LOGGING_STEPS = 5

print(f'CPU smoke test: {CPU_SMOKE_TEST}')
print(f'Model: {MODEL_NAME}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Max samples: train={MAX_TRAIN_SAMPLES}, val={MAX_VAL_SAMPLES}, test={MAX_TEST_SAMPLES}')

## 7. Fine-tune DNABERT-2

This calls the repository training script using the CSV files. Progress bars and step logs are enabled. In normal mode, the script saves the best checkpoint and final model under `training/output/dnabert2-clinvar-grch38/final_model`.

In [ ]:
force_cpu_arg = '--force-cpu' if FORCE_CPU else ''

!python training/scripts/train_dnabert2_classifier.py \
  --train-csv "{TRAIN_CSV}" \
  --val-csv "{VAL_CSV}" \
  --test-csv "{TEST_CSV}" \
  --output-dir "{OUTPUT_DIR}" \
  --model-name "{MODEL_NAME}" \
  --max-length {MAX_LENGTH} \
  --min-sequence-length {MIN_SEQUENCE_LENGTH} \
  --max-train-samples {MAX_TRAIN_SAMPLES} \
  --max-val-samples {MAX_VAL_SAMPLES} \
  --max-test-samples {MAX_TEST_SAMPLES} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --learning-rate {LEARNING_RATE} \
  --logging-steps {LOGGING_STEPS} \
  {force_cpu_arg}

## 8. Inspect Saved Model Metadata

In [ ]:
import json

metadata_path = OUTPUT_DIR / 'final_model' / 'variant_risk_metadata.json'
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text())
    print(json.dumps(metadata, indent=2))
else:
    print(f'Metadata not found yet: {metadata_path}')

## 9. Export Model Artifact

Zip the final model directory so you can download it from Colab and use it with the FastAPI backend later.

In [ ]:
ZIP_PATH = PROJECT_ROOT / 'dnabert2-clinvar-grch38-final-model.zip'
!cd "{PROJECT_ROOT}" && zip -qr "{ZIP_PATH}" training/output/dnabert2-clinvar-grch38/final_model
print(f'Wrote {ZIP_PATH}')

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except Exception:
    print('Download helper is only available inside Google Colab.')